# Total Site: Pulp Mill Utility-System Comparison
This notebook asks one decision question: what changes when the same pulp mill is viewed as a direct-integration problem versus a total-process and total-site utility-system problem, and how can the rendered graphs and serialized graph payloads both be inspected?

## Case context
The packaged `pulp_mill.json` case represents a multi-area pulp mill with steam levels, hot-water utility interaction, and site-wide utility tradeoffs. The workflow is: solve the baseline case, compare direct, total-process, and total-site targets, identify which subzones drive the site picture, inspect the direct GCC against total-site graphs, extract the raw graph payloads, and then compare a local cogeneration screen against the site-level cogeneration screen.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display

from OpenPinch import PinchProblem
from OpenPinch.resources import copy_sample_case

PLOT_WIDTH = 720
PLOT_HEIGHT = 540

SEGMENT_COLOURS = {
    0: "#e66e6e",
    1: "#5ca5d9",
    2: "#C22323",
    3: "#244abd",
    4: "#7f7f7f",
    5: "#111111",
}


def display_plotly(fig, *, width: int = PLOT_WIDTH, height: int = PLOT_HEIGHT):
    fig = fig.to_dict()
    figure = go.Figure(fig)
    figure.update_layout(width=width, height=height, autosize=False)
    display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))


def get_target_row(summary_frame: pd.DataFrame, target_name: str) -> pd.Series:
    row = summary_frame.loc[summary_frame["Target"] == target_name]
    if row.empty:
        raise KeyError(f"Target {target_name!r} was not found in the summary output.")
    return row.iloc[0]


def get_graph_by_type(graph_payload: dict, target_name: str, graph_type: str) -> dict:
    graph_set = graph_payload[target_name]
    for graph in graph_set["graphs"]:
        if graph.get("type") == graph_type:
            return graph
    raise KeyError(f"Graph type {graph_type!r} was not found for target {target_name!r}.")


def figure_from_graph_payload(graph: dict, *, title: str | None = None) -> go.Figure:
    fig = go.Figure()
    legend_seen: set[str] = set()

    for segment in graph.get("segments", []):
        points = segment.get("data_points", [])
        if not points:
            continue
        x_vals = [point["x"] for point in points]
        y_vals = [point["y"] for point in points]
        series = segment.get("title", "Segment")
        show_legend = series not in legend_seen
        legend_seen.add(series)

        line_style = {
            "color": SEGMENT_COLOURS.get(segment.get("colour"), "#333333"),
            "width": 2,
        }
        if len(x_vals) > 1 and all(abs(x - x_vals[0]) <= 1e-9 for x in x_vals[1:]):
            line_style["dash"] = "dash"

        fig.add_trace(
            go.Scatter(
                x=x_vals,
                y=y_vals,
                mode="lines",
                name=series,
                legendgroup=series,
                showlegend=show_legend,
                line=line_style,
                hovertemplate=(
                    f"{series}<br>Heat Flow / kW: %{{x}}<br>"
                    "Temperature / °C: %{y}<extra></extra>"
                ),
            )
        )

    fig.update_layout(
        title=title or graph.get("type", "Graph Payload"),
        xaxis_title="Heat Flow / kW",
        yaxis_title="Temperature / °C",
        template="plotly_white",
        hovermode="closest",
        legend={
            "title": "Click to toggle",
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "left",
            "x": 0,
        },
        margin={"l": 60, "r": 40, "t": 80, "b": 60},
    )
    return fig

In [ ]:
workspace = TemporaryDirectory()
work_dir = Path(workspace.name)
case_path = copy_sample_case(
    "pulp_mill.json",
    work_dir / "pulp_mill.json",
)

In [ ]:
problem = PinchProblem(source=case_path)
problem.target()
summary = problem.summary_frame()
catalog = problem.plot.catalog()
catalog

In [ ]:
graph_payload = problem.plot.get_graph_data()

direct_target_name = "Site/Direct Integration"
total_process_target_name = "Site/Total Process Target"
total_site_target_name = "Site/Total Site Target"

target_order = [
    direct_target_name,
    total_process_target_name,
    total_site_target_name,
]
target_labels = {
    direct_target_name: "Direct Integration",
    total_process_target_name: "Total Process Target",
    total_site_target_name: "Total Site Target",
}

## Base solve and graph inventory
Start by confirming the scope ladder and the graph families that are already available after one solve. The direct-integration target gives the local process view, while the total-site target introduces the site-wide utility graphs.

In [ ]:
catalog.loc[
    catalog["Target"].isin([direct_target_name, total_site_target_name]),
    ["Target", "Graph Type", "Graph Name"],
].drop_duplicates().reset_index(drop=True)
catalog

## Compare direct, total-process, and total-site scopes
Read these three rows first. They answer whether the same plant looks materially different once utility-system interaction is reconciled at the total-process and total-site levels.

In [ ]:
target_rows = summary.loc[
    summary["Target"].isin(target_order),
    [
        "Target",
        "Hot Utility Target",
        "Cold Utility Target",
        "Heat Recovery",
        "Hot Pinch",
        "Cold Pinch",
    ],
].copy()
target_rows["Scope"] = target_rows["Target"].map(target_labels)
target_rows = target_rows.set_index("Target").loc[target_order].reset_index(drop=True)
target_rows

In [ ]:
scope_rows = {row["Scope"]: row for _, row in target_rows.iterrows()}
comparison_pairs = [
    ("Total Process - Direct", "Total Process Target", "Direct Integration"),
    ("Total Site - Direct", "Total Site Target", "Direct Integration"),
]

scope_deltas = []
for label, lhs, rhs in comparison_pairs:
    left = scope_rows[lhs]
    right = scope_rows[rhs]
    scope_deltas.append(
        {
            "Comparison": label,
            "Hot Utility Delta": left["Hot Utility Target"] - right["Hot Utility Target"],
            "Cold Utility Delta": left["Cold Utility Target"] - right["Cold Utility Target"],
            "Heat Recovery Delta": left["Heat Recovery"] - right["Heat Recovery"],
            "Hot Pinch Delta": left["Hot Pinch"] - right["Hot Pinch"],
            "Cold Pinch Delta": left["Cold Pinch"] - right["Cold Pinch"],
        }
    )

scope_delta_frame = pd.DataFrame(scope_deltas)
scope_delta_frame

In [ ]:
target_metrics = target_rows.melt(
    id_vars="Scope",
    value_vars=["Hot Utility Target", "Cold Utility Target", "Heat Recovery"],
    var_name="Metric",
    value_name="Value",
)

metric_colours = {
    "Hot Utility Target": "#c0392b",
    "Cold Utility Target": "#2471a3",
    "Heat Recovery": "#138d75",
}

scope_comparison_fig = go.Figure()
for metric in ["Hot Utility Target", "Cold Utility Target", "Heat Recovery"]:
    metric_rows = target_metrics.loc[target_metrics["Metric"] == metric]
    scope_comparison_fig.add_bar(
        name=metric,
        x=metric_rows["Scope"],
        y=metric_rows["Value"],
        marker_color=metric_colours[metric],
        text=[f"{value:,.0f}" for value in metric_rows["Value"]],
        textposition="outside",
    )

scope_comparison_fig.update_layout(
    barmode="group",
    title="Utility targets and heat recovery across scope levels",
    xaxis_title="Scope level",
    yaxis_title="kW",
)
display_plotly(scope_comparison_fig, height=620)

## Which zones drive the site picture?
Before reading total-site graphs, look at the direct-integration zone rows. This shows which areas dominate hot utility demand, cold utility demand, and recovered heat before those effects are aggregated into a site-level utility system answer.

In [ ]:
zone_rows = summary.loc[
    summary["Target"].str.endswith("/Direct Integration")
    & (summary["Target"] != direct_target_name),
    ["Target", "Hot Utility Target", "Cold Utility Target", "Heat Recovery"],
].copy()
zone_rows["Zone"] = zone_rows["Target"].str.replace("/Direct Integration", "", regex=False)
zone_rows["Net Utility Demand"] = (
    zone_rows["Hot Utility Target"] + zone_rows["Cold Utility Target"]
)

leader_tables = []
for metric in ["Hot Utility Target", "Cold Utility Target", "Heat Recovery"]:
    leaders = zone_rows.nlargest(5, metric)[["Zone", metric]].reset_index(drop=True)
    leaders.index = leaders.index + 1
    leaders = leaders.rename_axis("Rank").reset_index()
    leaders["Metric"] = metric
    leader_tables.append(leaders[["Metric", "Rank", "Zone", metric]])

zone_leaders = pd.concat(leader_tables, ignore_index=True)
zone_leaders

## Render the key graphs in scope order
Read the local-process GCC first, then move to the total-site profiles and the Site Utility Grand Composite Curve. The direct GCC explains local bottlenecks; the total-site graphs explain utility-system leverage across the full site.

In [ ]:
direct_gcc = problem.plot.grand_composite_curve(zone_name=direct_target_name)
direct_gcc

In [ ]:
total_site_profiles_payload = get_graph_by_type(
    graph_payload,
    total_site_target_name,
    "Total Site Profiles",
)
total_site_profiles = figure_from_graph_payload(
    total_site_profiles_payload,
    title="Total Site Profiles",
)
display_plotly(total_site_profiles)

In [ ]:
sugcc_payload = get_graph_by_type(
    graph_payload,
    total_site_target_name,
    "Site Utility Grand Composite Curve",
)
sugcc = figure_from_graph_payload(
    sugcc_payload,
    title="Site Utility Grand Composite Curve",
)
display_plotly(sugcc)

## Extract the serialized graph payload
Rendered figures are one view. The underlying payload is another. `problem.plot.get_graph_data()` returns a target-keyed dictionary where each target entry contains zone metadata and a list of graph dictionaries. That payload is the surface to inspect when you want to reuse graph data outside the notebook.

In [ ]:
direct_gcc_payload = get_graph_by_type(
    graph_payload,
    direct_target_name,
    "Grand Composite Curve",
)

graph_payload_overview = pd.DataFrame(
    [
        {
            "Target": target_name,
            "Zone": graph_set["zone_name"],
            "Zone Address": graph_set["zone_address"],
            "Graph Count": len(graph_set["graphs"]),
            "Graph Types": ", ".join(graph["type"] for graph in graph_set["graphs"]),
        }
        for target_name, graph_set in graph_payload.items()
        if target_name in {direct_target_name, total_site_target_name}
    ]
)
graph_payload_overview

In [ ]:
selected_payloads = {
    "Direct GCC": direct_gcc_payload,
    "Total Site Profiles": total_site_profiles_payload,
    "Site Utility Grand Composite Curve": sugcc_payload,
}

payload_structure = pd.DataFrame(
    [
        {
            "Graph": label,
            "Graph Type": payload["type"],
            "Segment Count": len(payload.get("segments", [])),
            "First Segment Title": payload["segments"][0].get("title") if payload.get("segments") else None,
            "First Segment Keys": ", ".join(sorted(payload["segments"][0].keys())) if payload.get("segments") else None,
        }
        for label, payload in selected_payloads.items()
    ]
)
payload_structure

## Local versus site cogeneration screens
Run one local-zone screen and one total-site screen to make the scope difference explicit. The total-site result should be read as the main utility-system opportunity screen because it already reflects the reconciled site utility picture.

In [ ]:
cogen_bleach = problem.target.cogeneration(zone_name="Bleaching")
cogeneration_target = problem.target.cogeneration(
    options={
        "DO_TURBINE_WORK": True,
        "base_target_type": "Total Site Target",
    }
)

cogeneration_comparison = pd.DataFrame(
    [
        {
            "Screen": "Bleaching local screen",
            "Target": cogen_bleach.name,
            "Hot Utility Target": cogen_bleach.hot_utility_target,
            "Cold Utility Target": cogen_bleach.cold_utility_target,
            "Heat Recovery": cogen_bleach.heat_recovery_target,
            "Power Cogeneration Target": cogen_bleach.work_target,
            "Turbine Efficiency Target": cogen_bleach.turbine_efficiency_target,
        },
        {
            "Screen": "Total-site screen",
            "Target": cogeneration_target.name,
            "Hot Utility Target": cogeneration_target.hot_utility_target,
            "Cold Utility Target": cogeneration_target.cold_utility_target,
            "Heat Recovery": cogeneration_target.heat_recovery_target,
            "Power Cogeneration Target": cogeneration_target.work_target,
            "Turbine Efficiency Target": cogeneration_target.turbine_efficiency_target,
        },
    ]
)
cogeneration_comparison

## Interpretation
In this pulp mill, the direct-integration row gives the local process answer, but the total-process and total-site rows reveal how much that answer changes once utility-system interaction is reconciled. The direct GCC explains process bottlenecks; the total-site profiles show the broader hot/cold utility balance; the SUGCC is the graph to read when judging steam-system leverage and whether a site-wide utility change is worth deeper study. The serialized graph payload then gives the same graph families in a reusable data form keyed by target and zone metadata. Finally, the cogeneration comparison reinforces the scope rule: a local-zone screen can be informative, but the total-site screen is the right starting point for site utility-system decisions.